## BƯỚC 1: Cài đặt thư viện

In [ ]:
!pip install -q ultralytics opencv-python lapx

## BƯỚC 2: Upload Video và file cấu hình ROI
*(Chọn 2 file: `demo_clip.mp4` và `roi_config.json` từ máy tính)*

In [ ]:
from google.colab import files
print('Vui lòng chọn demo_clip.mp4 và roi_config.json:')
uploaded = files.upload()

## BƯỚC 3: Chạy Pipeline v1.1

In [ ]:
import os
import csv
import json
import cv2
import numpy as np
from ultralytics import YOLO

# --- CẤU HÌNH V1.1 ---
VIDEO_PATH = "demo_clip.mp4"
OUTPUT_VIDEO = "output_demo_v1.mp4"
CSV_PATH = "violations_yolov8s_0.5.csv"  # Format: violations_{model}_{N}.csv
ROI_CONFIG_PATH = "roi_config.json"
CROP_DIR = "crop"
FULL_DIR = "full"

MODEL_NAME = "yolov8s.pt"
CONF_THRES = 0.10
IOU_THRES = 0.50
IMGSZ = 1280

# COCO Classes: 2:car, 3:motorcycle, 5:bus, 7:truck
TRACKED_CLASSES = [2, 3, 5, 7]
VIOLATING_CLASS_ID = 3

VIOLATION_TIME_SEC = 0.3
LOST_TRACK_TOLERANCE = 3  # Cho phép mất dấu 3 khung hình

def run_pipeline():
    os.makedirs(CROP_DIR, exist_ok=True)
    os.makedirs(FULL_DIR, exist_ok=True)

    # Đọc cấu hình ROI từ json
    try:
        with open(ROI_CONFIG_PATH, "r") as f:
            roi_data = json.load(f)
            roi_polygon = np.array(roi_data["roi_polygon"], dtype=np.int32)
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file {ROI_CONFIG_PATH}.")
        print("Vui lòng tạo file roi_config.json với cấu trúc ví dụ:")
        print('{"roi_polygon": [[300, 250], [950, 250], [1150, 680], [150, 680]]}')
        return
    except Exception as e:
        print(f"Lỗi đọc file roi_config.json: {e}")
        return

    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print(f"Lỗi: Không mở được video {VIDEO_PATH}")
        return

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    n_frames_thres = int(np.ceil(VIOLATION_TIME_SEC * fps))

    writer = cv2.VideoWriter(
        OUTPUT_VIDEO,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (width, height)
    )

    model = YOLO(MODEL_NAME)
    
    # State tracking
    track_states = {} # track_id -> dict
    logged_tracks = set()
    violation_records = []
    violation_id = 1
    frame_id = 0

    print(f"Đang chạy {MODEL_NAME} | ROI: {roi_polygon.tolist()} | Ngưỡng: {n_frames_thres} khung")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_id += 1
        current_sec = round(frame_id / fps, 2)
        annotated_frame = frame.copy()

        cv2.polylines(annotated_frame, [roi_polygon], isClosed=True, color=(0, 255, 255), thickness=2)

        results = model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            classes=TRACKED_CLASSES,
            conf=CONF_THRES,
            iou=IOU_THRES,
            imgsz=IMGSZ,
            verbose=False
        )
        
        active_track_ids_in_frame = set()

        if results and results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            confs = results[0].boxes.conf.cpu().tolist()
            cls_ids = results[0].boxes.cls.int().cpu().tolist()

            for box, track_id, conf, cls_id in zip(boxes, track_ids, confs, cls_ids):
                active_track_ids_in_frame.add(track_id)
                x1, y1, x2, y2 = map(int, box)
                
                # Vẽ box xanh mặc định cho tất cả xe
                cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(annotated_frame, f"ID:{track_id} C:{cls_id}", (x1, max(15, y1 - 5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

                if cls_id != VIOLATING_CLASS_ID:
                    continue # Bỏ qua nếu không phải xe máy

                cx = int((x1 + x2) / 2)
                cy = int(y2)
                inside_roi = cv2.pointPolygonTest(roi_polygon, (cx, cy), False) >= 0

                if track_id not in track_states:
                    track_states[track_id] = {
                        "consecutive_frames": 0,
                        "lost_frames": 0,
                        "start_timestamp": 0.0,
                        "start_frame_id": 0,
                        "conf_sum": 0.0,
                        "is_violating": False
                    }

                state = track_states[track_id]

                if inside_roi:
                    if state["consecutive_frames"] == 0:
                        state["start_timestamp"] = current_sec
                        state["start_frame_id"] = frame_id
                    
                    state["consecutive_frames"] += 1
                    state["lost_frames"] = 0
                    state["conf_sum"] += conf
                else:
                    state["consecutive_frames"] = 0
                    state["lost_frames"] = 0
                    state["conf_sum"] = 0.0

                is_violating_now = (state["consecutive_frames"] >= n_frames_thres)
                
                if is_violating_now or track_id in logged_tracks:
                    color = (0, 0, 255)
                    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), color, 2)
                    cv2.circle(annotated_frame, (cx, cy), 4, color, -1)
                    cv2.putText(annotated_frame, f"ID:{track_id} VI PHAM", (x1, max(15, y1 - 5)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

                if is_violating_now and track_id not in logged_tracks:
                    logged_tracks.add(track_id)
                    
                    crop_filename = f"crop_{os.path.basename(VIDEO_PATH)}_t{track_id}_f{frame_id}.jpg"
                    full_filename = f"full_{os.path.basename(VIDEO_PATH)}_t{track_id}_f{frame_id}.jpg"
                    
                    crop_filepath = os.path.join(CROP_DIR, crop_filename)
                    full_filepath = os.path.join(FULL_DIR, full_filename)
                    
                    # Cắt ảnh crop đảm bảo không văng ra ngoài khung hình
                    h, w = frame.shape[:2]
                    y1_c = max(0, y1 - 20)
                    y2_c = min(h, y2 + 20)
                    x1_c = max(0, x1 - 20)
                    x2_c = min(w, x2 + 20)
                    crop_img = frame[y1_c:y2_c, x1_c:x2_c]
                    
                    cv2.imwrite(crop_filepath, crop_img)
                    cv2.imwrite(full_filepath, annotated_frame)
                    
                    avg_conf = state["conf_sum"] / state["consecutive_frames"]
                    
                    violation_records.append({
                        "id": violation_id,
                        "video_name": os.path.basename(VIDEO_PATH),
                        "track_id": track_id,
                        "vehicle_type": "motorcycle",
                        "timestamp_sec": f"{state['start_timestamp']:.2f}",
                        "frame_id": state["start_frame_id"],
                        "confidence": f"{avg_conf:.2f}",
                        "crop_path": f"{CROP_DIR}/{crop_filename}",
                        "full_path": f"{FULL_DIR}/{full_filename}"
                    })
                    violation_id += 1

        # Xử lý lost track (dung sai)
        for t_id, state in list(track_states.items()):
            if t_id not in active_track_ids_in_frame:
                if state["consecutive_frames"] > 0:
                    state["lost_frames"] += 1
                    if state["lost_frames"] >= LOST_TRACK_TOLERANCE:
                        state["consecutive_frames"] = 0
                        state["conf_sum"] = 0.0

        writer.write(annotated_frame)

    cap.release()
    writer.release()

    fieldnames = ["id", "video_name", "track_id", "vehicle_type", "timestamp_sec", "frame_id", "confidence", "crop_path", "full_path"]
    with open(CSV_PATH, mode="w", newline="", encoding="utf-8") as f:
        csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
        csv_writer.writeheader()
        csv_writer.writerows(violation_records)

    print(f"Hoàn tất v1.1! Video: {OUTPUT_VIDEO} | Số vi phạm: {len(violation_records)} | Đã lưu vào {CSV_PATH}")

if __name__ == "__main__":
    run_pipeline()


## BƯỚC 4: Đóng gói và Tải kết quả về máy

In [ ]:
import os
from google.colab import files

# Nén toàn bộ kết quả vào 1 file zip
!zip -r ketqua_GD1.zip output_demo_v1.mp4 violations_yolov8s_0.5.csv crop/ full/

# Tự động tải về máy
if os.path.exists('ketqua_GD1.zip'):
    files.download('ketqua_GD1.zip')
else:
    print('Lỗi: Không tìm thấy file kết quả để tải về.')